# Deep RecSys Course
## Домашнее задание 2: Нейросетевое ранжирование
### ФИО: Клюев Лев

В этом задании вы изучите способы кодирования вещественных и категориальных признаков, а затем обучите multi-task нейросетевую ранжирующую модель на основе DCN V2 со смесью экспертов.

Модель будет одновременно решать две задачи:
- предсказание полного прослушивания трека;
- предсказание лайка на трек.

Разбалловка

1. Подготовка датасета (1 балл)  
2. Обучение бейзлайнов и оценка качества (1 балл)  
3. Concat+MLP модель (3 балла)  
4. DCN-v2 (3 балла)  
5. Многоголовость (2 балла)  

# 1. Подготовка датасета (1 балл)

## yambda-50m-lag-features

Мы будем использовать датасет с уже заранее посчитанными признаками по Yambda.

Как был построен этот датасет:

1. За основу взят датасет Yambda `flat/50m multi_event`.
2. Из него были оставлены только неорганические события, то есть события, полученные с рекомендательных поверхностей, например из Моей Волны в Яндекс Музыке.
3. `is_skip = True`, если пользователь прослушал меньше половины трека.
4. `is_full_play = True`, если пользователь прослушал более 95% трека.
5. В Yambda лайк и прослушивание — это разные события. При этом модель обучается на событиях прослушивания: именно для них считаются признаки, и именно для них модель должна предсказывать таргеты. Поэтому лайки нужно атрибуцировать прослушиваниям, то есть понять, к какому именно прослушиванию относится каждый лайк. Для этого каждый лайк привязывается к ближайшему по времени прослушиванию того же `(uid, item_id)` в пределах 24 часов. Если подходящих прослушиваний два, выбирается ближайшее; при равенстве выбирается предыдущее. После этого `is_like = True` у прослушивания, если к нему был привязан хотя бы один лайк.
6. После этого по набору данных вычисляются вещественные счётчики. Они делятся на три типа: пользовательские, трековые и кросс-счётчики. Важно, что для каждого прослушивания с `timestamp` все счётчики считаются не в момент самого события, а в момент `timestamp - lag_seconds`, где `lag_seconds = 15 минут`. Такая задержка нужна, чтобы не допустить ликов при обучении и оценке. Идея в том, что при предсказании для текущего прослушивания модель не должна использовать информацию, которая появилась только после этого события или слишком близко к нему и в реальной системе ещё не успела бы попасть в признаки. Это особенно важно в нашей постановке, потому что далее качество будет оцениваться на соседних по времени прослушиваниях. Значит, при сравнении текущего трека с предыдущим нельзя допустить, чтобы признаки текущего объекта уже содержали фидбек на предыдущий трек, который к моменту предсказания ещё не должен был быть доступен. Иначе модель фактически будет подглядывать в ответ, а качество окажется завышенным. Кроме того, такой лаг лучше соответствует тому, как признаки обычно обновляются в реальных рекомендательных системах: не мгновенно, а с некоторой задержкой.
7. `is_like` и `is_full_play` используются как таргеты в задаче multi-task learning.

Код построения датасета можно посмотреть здесь: https://huggingface.co/datasets/matfu21/yambda-50m-lag-features/blob/main/event_processor.py#L296

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import sys

sys.path.insert(0, "/kaggle/input/datasets/levklyuev/i-need")

In [3]:
!pip install polars huggingface_hub datasets catboost torch

In [4]:
import warnings
warnings.filterwarnings("ignore")

import gc
import polars as pl
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="matfu21/yambda-50m-lag-features",
    repo_type="dataset",
    filename="listens.parquet",
)
listens = pl.read_parquet(path)

## Оставляем только предпочтительные пары

В задачах ранжирования модель обычно обучается не на всех событиях подряд, а на специально отобранных объектах, которые несут сигнал о пользовательских предпочтениях. Это особенно важно для относительно небольших ранжирующих моделей. Если обучать такую модель на всех примерах подряд, она будет в большей степени учиться предсказывать глобальные вероятности лайка или полного прослушивания, чем правильно упорядочивать близкие по времени объекты. В результате качество именно на тех локальных попарных сравнениях, которые нас реально интересуют, может ухудшаться. Поэтому на практике датасеты для обучения ранжирования часто семплируются так, чтобы оставлять только примеры с полезным фидбеком.

При этом важно понимать, что это скорее инженерный компромисс, а не фундаментальное ограничение. Если бы у нас были очень большие модели с высокой выразительной способностью и без жёстких ограничений по вычислениям, то в идеале действительно хотелось бы учиться на всех доступных событиях. Такая модель могла бы одновременно хорошо восстанавливать глобальные закономерности и при этом качественно моделировать локальные предпочтения. Но в реальных рекомендательных системах ранжирующие модели обычно остаются относительно компактными, поэтому им полезно помогать фокусироваться именно на тех примерах, которые наиболее важны для ранжирования.

Почему ранжирующие модели обычно относительно небольшие: такие модели работают на финальном этапе ранжирования, где нужно очень быстро пересчитать score для большого числа кандидатов. У системы обычно есть жёсткие ограничения по latency, памяти и числу запросов в секунду. Поэтому слишком большие модели здесь использовать трудно: они будут слишком дорогими и медленными в продакшене. Именно из-за этого особенно важно, чтобы маленькая или средняя по размеру модель тратила свою выразительную способность на действительно полезные примеры, а не на предсказание усреднённых глобальных вероятностей.

В этом задании вам нужно оставить только те прослушивания, которые входят хотя бы в одну локальную пару с различающимся фидбеком. Идея такая: если у текущего трека и у соседнего по времени трека разный сигнал, то такая пара помогает обучать ранжирование. Если же и слева, и справа фидбек одинаковый, то текущее событие менее полезно для этой постановки.

### Что нужно сделать

Для каждого события нужно:
1. посмотреть на предыдущее событие того же пользователя;
2. посмотреть на следующее событие того же пользователя;
3. проверить, отличается ли `is_like` или `is_full_play` хотя бы с одной из сторон;
4. оставить только те строки, для которых такое отличие есть.

### Важная деталь про сортировку

Перед тем как смотреть на соседние события, данные нужно отсортировать по:
- `uid`,
- `timestamp`,
- `row_id`.

Поле `row_id` нужно для детерминированности. У одного и того же пользователя могут встретиться события с одинаковым `timestamp`, и тогда сортировка только по `uid` и `timestamp` не задаёт однозначный порядок. Добавление `row_id` делает этот порядок фиксированным и воспроизводимым.

In [5]:
def extract_preference_pairs(df: pl.DataFrame) -> pl.DataFrame:
  return (
      df
      .with_row_index("row_id")
      .sort(["uid", "timestamp", "row_id"])
      .filter(
          (pl.col("is_like") != pl.col("is_like").shift(1).over("uid")) |
          (pl.col("is_like") != pl.col("is_like").shift(-1).over("uid")) |
          (pl.col("is_full_play") != pl.col("is_full_play").shift(1).over("uid")) |
          (pl.col("is_full_play") != pl.col("is_full_play").shift(-1).over("uid"))
      )
      .drop("row_id")
  )
listens = extract_preference_pairs(listens)

## Набор данных Yambda

В этом задании мы будем использовать датасет Yambda: https://huggingface.co/datasets/yandex/yambda. Из него мы получим информацию об альбомах, к которым относится трек, а также об артистах этого трека. Эти данные понадобятся вам для построения мультивалентных признаков.

Ниже приведён класс из Hugging Face, который вам предстоит использовать.

In [6]:
from dataclasses import dataclass


@dataclass
class DatasetConfig:
    dataset_type: str = 'flat'
    dataset_size: str = '50m'
    interaction_name: str = 'multi_event'
    default_like_window_seconds: int = 24 * 60 * 60
    lag_seconds: int = 15 * 60

dataset_config = DatasetConfig()

In [7]:
from typing import Literal
from datasets import Dataset, DatasetDict, load_dataset


# YambdaDataset wrapper class (https://huggingface.co/datasets/yandex/yambda)
class YambdaDataset:
    INTERACTIONS = frozenset([
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ])

    def __init__(
        self,
        dataset_type: Literal["flat", "sequential"] = "flat",
        dataset_size: Literal["50m", "500m", "5b"] = "50m"
    ):
        assert dataset_type in {"flat", "sequential"}
        assert dataset_size in {"50m", "500m", "5b"}
        self.dataset_type = dataset_type
        self.dataset_size = dataset_size

    def interaction(self, event_type: Literal[
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ]) -> Dataset:
        assert event_type in YambdaDataset.INTERACTIONS
        return self._download(f"{self.dataset_type}/{self.dataset_size}", event_type)

    def audio_embeddings(self) -> Dataset:
        return self._download("", "embeddings")

    def album_item_mapping(self) -> Dataset:
        return self._download("", "album_item_mapping")

    def artist_item_mapping(self) -> Dataset:
        return self._download("", "artist_item_mapping")

    @staticmethod
    def _download(data_dir: str, file: str) -> Dataset:
        data = load_dataset("yandex/yambda", data_dir=data_dir, data_files=f"{file}.parquet")
        # Returns DatasetDict; extracting the only split
        assert isinstance(data, DatasetDict)
        return data["train"]


In [8]:
yambda_dataset = YambdaDataset(
    dataset_type=dataset_config.dataset_type,
    dataset_size=dataset_config.dataset_size
)
albums = yambda_dataset.album_item_mapping().to_polars()
artists = yambda_dataset.artist_item_mapping().to_polars()

## Добавляем мультивалентные признаки

Ранее мы загрузили данные `albums` и `artists`. Теперь на их основе нужно добавить в таблицу две новые колонки с мультивалентными категориальными признаками:
- `artist_ids` — список артистов, соответствующих данному треку;
- `album_ids` — список альбомов, соответствующих данному треку.

Таким образом, для каждого трека в этих колонках должен храниться не один идентификатор, а список идентификаторов.

In [9]:
def join_item_artist_album(
    listens: pl.DataFrame,
    artists: pl.DataFrame,
    albums: pl.DataFrame,) -> pl.DataFrame:

    arts = artists.group_by("item_id").agg(
        pl.col("artist_id").alias("artist_ids")
    )

    alb = albums.group_by("item_id").agg(
        pl.col("album_id").alias("album_ids")
    )

    listens = listens.join(arts, on="item_id", how="left").join(alb, on="item_id", how="left")
    return listens

listens = join_item_artist_album(listens, artists, albums)
print(listens.shape)
del artists, albums
gc.collect()

(6885472, 25)


86

## Разбиваем данные на обучающую и тестовую выборки

Далее вам нужно написать функцию, которая делит датасет на две части по времени:
- в `train` попадают все события, кроме последних `test_last_seconds` секунд;
- в `test` попадают события из последних `test_last_seconds` секунд, включая границу.

В этом задании в качестве тестовой выборки мы будем использовать последние 30 дней.

In [10]:
def temporal_train_test_split(
    df: pl.DataFrame,
    test_last_seconds: float,
    time_column: str = "timestamp",) -> tuple[pl.DataFrame, pl.DataFrame]:

    mx_time = df.select(pl.col("timestamp").max()).item()
    train = df.filter(pl.col("timestamp") < mx_time - test_last_seconds)
    test = df.filter(pl.col("timestamp") >= mx_time - test_last_seconds)

    return (train, test)

train_listens, test_listens = temporal_train_test_split(listens, test_last_seconds=30 * 24 * 60 * 60)
del listens
gc.collect()

0

# 2. Обучение бейзлайнов и оценка качества (1 балл)

## Pairwise accuracy

Метрика, которой будет оцениваться качество вашего решения, называется `pairwise accuracy`.

Идея метрики такая: мы хотим проверить, умеет ли модель правильно упорядочивать соседние по времени события одного пользователя. Например, если в паре соседних треков на один был лайк, а на другой нет, то трек с лайком должен получить больший скор.

Функция получает на вход:
- `uids` — идентификаторы пользователей;
- `timestamps` — времена событий;
- `labels` — бинарные таргеты (`0` или `1`);
- `probs` — предсказанные моделью скоры;
- `session_gap_seconds` — максимальный допустимый разрыв по времени между соседними событиями.

Что нужно реализовать:
1. сгруппировать события по пользователям;
2. внутри каждого пользователя отсортировать события по времени;
3. рассмотреть только соседние пары;
4. отбросить пары, у которых разница по времени больше `session_gap_seconds`;
5. оставить только пары, у которых различаются `label`;
6. проверить, совпадает ли порядок по `probs` с порядком по `label`.

Вклад одной пары:
- 1, если модель ранжирует пару правильно;
- 0, если неправильно;
- 0.5, если скоры равны.

Формально, для соседней валидной пары $(i, i+1)$:

$$
m(i, i+1)=
\begin{cases}
1, & \text{если } (y_{i+1}-y_i)(s_{i+1}-s_i) > 0, \\
0.5, & \text{если } s_{i+1} = s_i, \\
0, & \text{иначе.}
\end{cases}
$$

Итоговая метрика:

$$
\text{PairwiseAccuracy} =
\frac{1}{|P|}\sum_{(i,i+1)\in P} m(i,i+1),
$$

где $P$ — множество всех валидных соседних пар по всем пользователям.

Почему мы смотрим именно на соседние пары: в рекомендательных задачах контекст пользователя быстро меняется. Если сравнивать два события, между которыми прошло много времени, такое сравнение уже может быть некорректным: пользователь был в другом состоянии, слушал другую музыку и вообще решал другую задачу. Поэтому мы сравниваем только локальные, соседние по времени события.

По этой же причине здесь не очень подходит ROC AUC. Эта метрика фактически сравнивает позитивные и негативные объекты между собой глобально, в том числе из совершенно разных моментов времени. В результате модель может получить хороший ROC AUC, даже если она плохо упорядочивает близкие по времени события внутри реальной пользовательской последовательности. `Pairwise accuracy` лучше отражает именно качество локального ранжирования, которое важно в этой задаче.

Дополнительно важно, что временной порог в метрике согласован с лагом, который использовался при построении признаков. В признаках мы используем задержку 15 минут, чтобы модель не могла опираться на слишком свежий фидбек, который в реальной системе ещё не успел бы попасть в фичи. В метрике используется тот же порог: если два соседних события находятся друг от друга дальше, чем на 15 минут, мы их не сравниваем. Это делает постановку более согласованной: и при построении признаков, и при оценке качества мы работаем в одном и том же локальном временном окне.

In [11]:
import numpy as np


def compute_pairwise_accuracy(
    uids: np.array,
    timestamps: np.array,
    labels: np.array,
    probs: np.array,
    session_gap_seconds: int = 15 * 60,):

    df = pl.DataFrame(
        {
            "uids": uids,
            "timestamp": timestamps,
            "labels": labels,
            "probs": probs,
        }
    )

    df = (
        df
        .with_row_index("row_id")
        .sort(["uids", "timestamp", "row_id"])
        .with_columns(
            next_time = pl.col("timestamp").shift(-1).over("uids"),
            next_lab = pl.col("labels").shift(-1).over("uids"),
            next_pro = pl.col("probs").shift(-1).over("uids"),
        )
        .filter(
            (pl.col("next_time") - pl.col("timestamp") <= session_gap_seconds) &
            (pl.col("labels") != pl.col("next_lab"))
        )
    )

    df = df.with_columns(
      score = pl
      .when((pl.col("next_pro") - pl.col("probs")) * (pl.col("next_lab") - pl.col("labels")) > 0)
      .then(1.0)
      .when((pl.col("next_pro") == pl.col("probs")))
      .then(0.5)
      .otherwise(0.0))

    accuracy = df.select(pl.col("score").mean()).item()
    if accuracy is None:
      return 0.0
    return accuracy

## Считаем метрики случайного бейзлайна

Перед обучением модели полезно посчитать качество простого случайного бейзлайна. Для этого сгенерируйте для каждого объекта в тестовой выборке случайный скор и посчитайте по нему обе метрики `pairwise accuracy`:
- по таргету `is_like`;
- по таргету `is_full_play`.

На этом шаге нужно:
1. взять тестовую выборку;
2. сгенерировать для каждого объекта случайное значение score;
3. вычислить `pair_accuracy_like`;
4. вычислить `pair_accuracy_full_play`.

В результате у вас должен получиться словарь `metrics` с двумя значениями метрик для случайного ранжирования.

In [12]:
test_listens = test_listens.with_columns(
    random_score=(
        pl.int_range(0, pl.len())
        .hash(seed=42)
        .cast(pl.Float64)
        / float(2**64 - 1)
    )
)
metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(test_listens.select("uid"), test_listens.select("timestamp"), test_listens.select("is_like").cast(pl.Int8), test_listens.select("random_score"),),
    "pair_accuracy_full_play": compute_pairwise_accuracy(test_listens.select("uid"), test_listens.select("timestamp"), test_listens.select("is_full_play").cast(pl.Int8), test_listens.select("random_score"),)
}

## Считаем метрики popularity baseline

Ещё один простой бейзлайн — ранжировать треки по их популярности в обучающей выборке. Для этого посчитайте, сколько раз каждый `item_id` встретился в `train`, и используйте это значение как score на `test`.

На этом шаге нужно:
1. по обучающей выборке посчитать популярность каждого трека (сколько раз он встретился);
2. присоединить эту популярность к объектам тестовой выборки;
3. для треков, которых не было в `train`, подставить `0`;
4. вычислить `pair_accuracy_like`;
5. вычислить `pair_accuracy_full_play`.

В результате у вас должен получиться словарь `metrics` с двумя значениями метрик для popularity baseline.

In [13]:
test_listens = test_listens.drop(
    [c for c in ["popularity", "popularity_right"] if c in test_listens.columns]
)

In [14]:
popul = train_listens.group_by("item_id").agg(
    pl.len().alias("popularity")
)
test_listens = (
    test_listens
    .join(popul, on="item_id", how="left")
    .with_columns(
        pl.col("popularity").fill_null(0)
    )
)
test_listens = test_listens.with_columns(
    pl.col("popularity") / pl.col("popularity").max()
    .alias("popularity")
)

metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        test_listens.select("uid"), test_listens.select("timestamp"), test_listens.select("is_like").cast(pl.Int8), test_listens.select("popularity"),
    ),
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        test_listens.select("uid"), test_listens.select("timestamp"), test_listens.select("is_full_play").cast(pl.Int8), test_listens.select("popularity"),
    )
}

## Обучаем CatBoost baseline

Теперь обучим ещё один бейзлайн — модель CatBoost. В отличие от предыдущих простых бейзлайнов, здесь модель уже будет использовать признаки из датасета и учиться предсказывать таргет по обучающей выборке.

На этом шаге нужно:
1. взять вещественные признаки `DENSE_COLUMNS` и категориальные признаки `SPARSE_COLUMNS`;
2. обучить `CatBoostClassifier` на задаче предсказания `is_full_play`;
3. получить предсказанные вероятности на тестовой выборке;
4. использовать вероятность полного прослушивания как score;
5. посчитать `pair_accuracy_like`;
6. посчитать `pair_accuracy_full_play`.

Обратите внимание, что модель обучается только на таргете `is_full_play`, но качество нужно измерить по обоим таргетам. Это позволит понять, насколько предсказание лайка переносится на задачу ранжирования по полному прослушиванию.

В результате у вас должен получиться словарь `metrics` с двумя значениями:
- `pair_accuracy_like`;
- `pair_accuracy_full_play`.

In [15]:
from catboost import CatBoostClassifier

DENSE_COLUMNS: tuple[str, ...] = (
    "user_lag_listen_cnt",
    "user_lag_like_cnt",
    "user_lag_full_play_cnt",
    "user_lag_skip_cnt",
    "item_lag_listen_cnt",
    "item_lag_like_cnt",
    "item_lag_full_play_cnt",
    "item_lag_skip_cnt",
    "ui_lag_listen_cnt",
    "ui_lag_like_cnt",
    "ui_lag_full_play_cnt",
    "ui_lag_skip_cnt",
    "user_lag_avg_played_ratio",
    "item_lag_avg_played_ratio",
    "ui_lag_avg_played_ratio",
)
MULTIVALENT_COLUMNS: tuple[str, ...] = ("artist_ids", "album_ids")
SPARSE_COLUMNS = ("uid", "item_id")
LABEL_COLUMNS = ("is_like", "is_full_play")

X_train = train_listens.select(DENSE_COLUMNS + SPARSE_COLUMNS)
X_test = test_listens.select(DENSE_COLUMNS + SPARSE_COLUMNS)
model = CatBoostClassifier(iterations=50)

model.fit(X_train, train_listens["is_full_play"], cat_features=list(SPARSE_COLUMNS))
scores = model.predict_proba(X_test)[:, 1]

test_listens = test_listens.with_columns(
    score=pl.Series(scores)
)

metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        test_listens.select("uid"), test_listens.select("timestamp"), test_listens.select("is_like").cast(pl.Int8), test_listens.select("score"),
    ),
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        test_listens.select("uid"), test_listens.select("timestamp"), test_listens.select("is_full_play").cast(pl.Int8), test_listens.select("score"),)
}

Learning rate set to 0.5
0:	learn: 0.6791001	total: 905ms	remaining: 44.4s
1:	learn: 0.6738385	total: 1.58s	remaining: 37.9s
2:	learn: 0.6713083	total: 2.1s	remaining: 32.9s
3:	learn: 0.6695331	total: 2.91s	remaining: 33.5s
4:	learn: 0.6686613	total: 3.37s	remaining: 30.3s
5:	learn: 0.6679903	total: 3.95s	remaining: 29s
6:	learn: 0.6676868	total: 4.48s	remaining: 27.5s
7:	learn: 0.6673333	total: 5.02s	remaining: 26.4s
8:	learn: 0.6670399	total: 5.57s	remaining: 25.4s
9:	learn: 0.6668030	total: 6.12s	remaining: 24.5s
10:	learn: 0.6666165	total: 6.63s	remaining: 23.5s
11:	learn: 0.6664187	total: 7.15s	remaining: 22.7s
12:	learn: 0.6662333	total: 7.68s	remaining: 21.8s
13:	learn: 0.6661010	total: 8.2s	remaining: 21.1s
14:	learn: 0.6660228	total: 8.7s	remaining: 20.3s
15:	learn: 0.6659388	total: 9.22s	remaining: 19.6s
16:	learn: 0.6658235	total: 9.73s	remaining: 18.9s
17:	learn: 0.6657595	total: 10.3s	remaining: 18.3s
18:	learn: 0.6656654	total: 10.8s	remaining: 17.6s
19:	learn: 0.6655659	

# 3. Concat+MLP модель (3 балла)

В этом задании вам предстоит реализовать первое нейросетевое решение.

Идея модели очень простая:
1. вещественные признаки кодируются с помощью `PiecewiseLinearEncoder`;
2. категориальные и мультивалентные признаки кодируются с помощью `Multisize Unified Embeddings`;
3. все полученные представления конкатенируются в один вектор;
4. этот вектор подаётся в простую полносвязную сеть с `ReLU` в качестве нелинейности.

На этом этапе модель предсказывает только вероятность полного прослушивания, то есть решается одна бинарная задача. Пока что речь о multi-task learning и нескольких головах не идёт.

Таким образом, вам нужно реализовать модель, которая:
- получает dense-, sparse- и multivalent-признаки;
- кодирует каждый тип признаков своим энкодером;
- объединяет все представления;
- пропускает результат через `MLP`;
- выдаёт скалярный score для предсказания полного прослушивания.

## Готовим датасет для обучения

Теперь нужно реализовать класс `RankerDataset`, который будет подготавливать батчи для обучения нейросетевой модели.

В этом задании батчевание происходит **на этапе самого датасета**, а не в `DataLoader`. Здесь это сделано специально: табличные данные уже лежат целиком в памяти, а модель сама по себе небольшая, поэтому узким местом легко становится именно подготовка батчей и перекладывание данных. Если формировать большие батчи на лету, то даже при больших `num_workers` и `prefetch_factor` загрузка данных может стать bottleneck. Поэтому здесь удобнее заранее нарезать датафрейм на батчи и хранить их в готовом виде внутри датасета.

Вам дан каркас класса `RankerDataset`. Нужно реализовать его так, чтобы в конструкторе датафрейм разбивался на батчи размера `batch_size`, а каждый батч преобразовывался в словарь фиксированной структуры.

### Что нужно сделать

1. Проверить, что `batch_size >= 1`.
2. Разбить входной `df` на последовательные батчи размера `batch_size`.
3. Для каждого батча собрать dense-, sparse- и multivalent-признаки, а также таргеты и мета-информацию.
4. Последовательно применить к каждому батчу все функции из `transforms`.
5. Сохранить готовые батчи внутри датасета.
6. Реализовать:
   - `__len__` — количество батчей;
   - `__getitem__` — получение батча по индексу.

### Формат батча

Каждый батч должен быть словарём со следующими ключами:
- `labels`
- `dense_features`
- `sparse_features`
- `multivalent_features`
- `meta`

#### `labels`

Словарь, где ключами являются названия таргетов из `label_columns`, а значениями — одномерные `torch.Tensor` длины `batch_size`.

Пример:

```python
{
    "is_like": ...,
    "is_full_play": ...,
}
```

#### `dense_features`
Тензор размера [batch_size, num_dense_features] типа torch.float32.

#### `sparse_features`
Словарь, где для каждого имени из sparse_columns хранится одномерный тензор длины batch_size.

Пример:

```python
{
    "uid": ...,
    "item_id": ...,
}
```

#### `multivalent_features`
Словарь, где для каждого мультивалентного признака нужно сохранить ещё один словарь с двумя полями:
-  values — один общий плоский тензор со всеми значениями из списков;
- lengths — тензор длин списков для каждого объекта в батче.

Пример:

```python
{
    "artist_ids": {
        "values": ...,
        "lengths": ...,
    },
    "album_ids": {
        "values": ...,
        "lengths": ...,
    },
}
```

Именно в таком формате мультивалентные признаки удобно потом агрегировать внутри модели.

#### `meta`
Словарь с технической информацией о батче:

```python
{
    "timestamp": ...,
    "uid": ...,
    "item_id": ...,
}
```

Что такое transforms

Аргумент transforms — это список функций, которые принимают батч и возвращают преобразованный батч. Эти преобразования нужно применять последовательно к каждому уже собранному батчу.

То есть логика такая:
1. вы собрали батч в нужном формате;
2.	прогнали его через все функции из transforms;
3.	сохранили итоговый батч в датасет.

Это позволит дальше независимо добавлять, например, перенос на устройство, нормализацию, переименование полей или любые другие постобработки.

Дополнительные замечания
- Dense-признаки и таргеты нужно привести к torch.float32.
- Для мультивалентных признаков lengths и values должны быть целочисленными тензорами.
- Последний батч может быть неполным.
- Пустые батчи сохранять не нужно.

В результате ваш RankerDataset должен возвращать готовые батчи, а не отдельные объекты.

In [16]:
import torch
import polars as pl
from typing import Any, Callable
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from collections import defaultdict

class RankerDataset(Dataset):
    def __init__(
        self,
        df: pl.DataFrame,
        transforms: list[Callable[[Any], Any]],
        label_columns: list[str],
        dense_columns: list[str],
        sparse_columns: list[str],
        multivalent_columns: list[str],
        batch_size: int,):

        if batch_size < 1:
          raise ValueError("batch должен быть хотя бы 1")

        self.batches = []
        num_rows = len(df)

        for start_idx in tqdm(range(0, num_rows, batch_size)):
            end_idx = min(start_idx + batch_size, num_rows)
            batch_df = df[start_idx:end_idx]

            batch_dict = defaultdict(dict)

            labels = batch_dict["labels"]
            for label_column in label_columns:
                labels[label_column] = torch.tensor(
                    batch_df[label_column].to_numpy(), dtype=torch.float32
                )

            batch_dict["dense_features"] = torch.tensor(
                batch_df.select(dense_columns).to_numpy(), dtype=torch.float32
            )

            sparse_features = batch_dict["sparse_features"]
            for sparse_column in sparse_columns:
                sparse_features[sparse_column] = torch.tensor(
                    batch_df[sparse_column].to_numpy(), dtype=torch.long
                )

            multivalent_features = batch_dict["multivalent_features"]
            for multivalent_column in multivalent_columns:
                values_list = []
                lengths_list = []

                for series in batch_df[multivalent_column]:
                    if series is None or len(series) == 0:
                        values_list.append(0)
                        lengths_list.append(0)
                    else:
                        values_list.extend(series)
                        lengths_list.append(len(series))

                multivalent_features[multivalent_column] = {
                    "values": torch.tensor(values_list, dtype=torch.long),
                    "lengths": torch.tensor(lengths_list, dtype=torch.long),
                }

            batch_dict["meta"] = {}
            for column in ["timestamp", "uid", "item_id"]:
                batch_dict["meta"][column] = torch.tensor(
                    batch_df[column].to_numpy(), dtype=torch.long
                )

            for transform in transforms:
                batch_dict = transform(batch_dict)

            self.batches.append(batch_dict)


    def __len__(self) -> int:
        return len(self.batches)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        return self.batches[idx]

## Multihash transform

Теперь реализуем `MultihashTransform` — преобразование, которое будет переводить sparse- и multivalent-признаки в набор хешей. Этот шаг нужен для дальнейшей работы с unified embeddings.

Идея здесь следующая: вместо того, чтобы хранить отдельную embedding-таблицу под каждый признак или использовать один-единственный хеш, мы отображаем каждый идентификатор сразу в несколько хешей с разными seed. В результате один и тот же исходный ID превращается в небольшой набор индексов. Такой подход предлагается в статье [Unified Embedding: Battle-Tested Feature Representations for Web-Scale ML Systems](https://arxiv.org/pdf/2305.12102) и позволяет компактнее работать с большим количеством категориальных признаков.

Вам дан каркас класса `MultihashTransform`. Нужно реализовать его так, чтобы он применял multihash-преобразование:
- к sparse-признакам;
- к multivalent-признакам.

### Что приходит на вход

На вход `__call__` подаётся `sample` — словарь батча, в котором уже есть поля:
- `sparse_features`;
- `multivalent_features`.

Именно их и нужно преобразовать.

### Что задаётся в конструкторе

- `sparse_features_config` — словарь вида  
  `{feature_name: [seed1, seed2, ...]}`  
  для sparse-признаков;
- `sparse_features_name` — имя поля в `sample`, где лежат sparse-признаки;
- `multivalent_features_config` — словарь такого же вида для multivalent-признаков;
- `multivalent_features_name` — имя поля в `sample`, где лежат multivalent-признаки;
- `cardinality` — размер общего хеш-пространства, то есть все индексы после хеширования должны лежать в диапазоне `[0, cardinality)`.

### Что нужно сделать

Для каждого sparse-признака:
1. взять тензор идентификаторов;
2. для каждого `seed` из конфига отдельно посчитать хеш;
3. привести хеш в диапазон `[0, cardinality)` с помощью взятия остатка;
4. сохранить результат обратно в `sample`.

Для каждого multivalent-признака:
1. взять плоский тензор `values`;
2. для каждого `seed` из конфига отдельно посчитать хеш;
3. привести хеш в диапазон `[0, cardinality)`;
4. сохранить результат обратно в `sample`, не меняя `lengths`.

### Какой формат должен получиться

Если исходный sparse-признак имел форму

```python
[batch_size]
```

то после multihash-преобразования он должен иметь форму

```python
[batch_size, num_hashes]
```

где num_hashes — число seed для этого признака.

Если values у multivalent-признака имели форму

```python
[num_values]
```

то после преобразования должно получиться

```python
[num_values, num_hashes]
```

Поле lengths у multivalent-признаков менять не нужно.

Важные замечания
- Для хеширования нужно использовать mmh3.
- Один и тот же исходный ID при одном и том же seed всегда должен давать один и тот же индекс.
- Разные seed должны давать разные хеш-проекции одного и того же ID.
- Все полученные индексы должны быть целыми числами из диапазона `[0, cardinality)`.
- Преобразование должно происходить in-place: нужно модифицировать sample и вернуть его обратно.

Итоговая идея такая: после MultihashTransform каждый категориальный ID заменяется не одним индексом, а набором индексов, полученных несколькими независимыми хеш-функциями.


In [17]:
!pip install mmh3

In [18]:
from typing import Any
import mmh3


class MultihashTransform:
    def __init__(
        self,
        sparse_features_config: dict,
        sparse_features_name: str,
        multivalent_features_config: dict,
        multivalent_features_name: str,
        cardinality: int,
    ):
        self.sparse_features_config = sparse_features_config
        self.sparse_features_name = sparse_features_name
        self.multivalent_features_config = multivalent_features_config
        self.multivalent_features_name = multivalent_features_name
        self.cardinality = cardinality


    def _apply_multihash(
        self, tensor: torch.Tensor, seeds: list
    ) -> torch.Tensor:
        assert tensor.device.type == "cpu"
        tensor_np = tensor.numpy()

        hashed_tensors = []
        for seed in seeds:
            hashed_np = np.array(
                [
                    mmh3.hash(str(int(x)), seed) % self.cardinality
                    for x in tensor_np.flatten()
                ]
            ).reshape(tensor_np.shape)
            hashed_tensors.append(torch.from_numpy(hashed_np))

        return torch.stack(hashed_tensors, dim=-1).to(tensor.device)

    def __call__(self, sample: dict[str, Any]) -> dict[str, Any]:
        sparse_features = sample[self.sparse_features_name]
        for feature_name, seeds in self.sparse_features_config.items():
            sparse_features[feature_name] = self._apply_multihash(
                sparse_features[feature_name], seeds
            )

        multivalent_features = sample[self.multivalent_features_name]
        for feature_name, seeds in self.multivalent_features_config.items():
            multivalent_features[feature_name]["values"] = (
                self._apply_multihash(
                    multivalent_features[feature_name]["values"], seeds
                )
            )

        return sample

## CategoricalEncoder

Теперь реализуем `CategoricalEncoder` — простой энкодер категориальных признаков. Его задача состоит в том, чтобы по входным индексам возвращать соответствующие embedding-векторы.

В конструктор класса передаётся объект `nn.Embedding`, который нужно сохранить внутри модуля. В методе `forward` на вход подаётся тензор `ids`, а на выходе должен получаться результат применения `embeddings(ids)`.

Иными словами, `CategoricalEncoder` — это тонкая обёртка над `nn.Embedding`.

### Что нужно реализовать

1. Сохранить переданный слой `embeddings`.
2. В `forward` применить его к тензору `ids`.
3. Вернуть получившийся тензор.

### Ожидаемое поведение

Если на вход подан тензор индексов формы

```python
[batch_size]
```

то на выходе должен получиться тензор формы

```python
[batch_size, embedding_dim]
```

Если на вход подан тензор формы

```python
[batch_size, num_hashes]
```

то на выходе должен получиться тензор формы

```python
[batch_size, num_hashes, embedding_dim]
```

То есть энкодер должен работать с произвольной входной формой так же, как обычный nn.Embedding.


In [19]:
import torch
import torch.nn as nn


class CategoricalEncoder(nn.Module):
    def __init__(self, embeddings: nn.Embedding):
        super().__init__()
        self.emb = embeddings

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        return self.emb(ids)

## MultivalentEncoder

Теперь реализуем `MultivalentEncoder` — энкодер для мультивалентных категориальных признаков. В отличие от обычного категориального признака, здесь каждому объекту соответствует не один ID, а список ID, например список артистов или альбомов.

В конструктор класса передаётся объект `nn.Embedding`, который нужно сохранить внутри модуля. В методе `forward` на вход подаются:
- `ids` — плоский тензор идентификаторов после `MultihashTransform`;
- `lengths` — длины списков для каждого объекта в батче.

Нужно получить embedding для каждого ID, а затем усреднить эмбеддинги внутри каждого списка. Для этого удобно использовать `torch.nn.functional.embedding_bag` с режимом `"mean"`.

### Что нужно реализовать

1. Сохранить переданный слой `embeddings`.
2. В `forward` агрегировать эмбеддинги по спискам с помощью среднего.
3. Вернуть тензор представлений объектов.

### Ожидаемое поведение

Если на вход подан:
- `ids` формы `[num_values, num_hashes]`,
- `lengths` формы `[batch_size]`,

то на выходе должен получиться тензор формы

```python
[batch_size, num_hashes, embedding_dim]
```

То есть для каждого объекта в батче нужно получить по одному усреднённому embedding-представлению для каждого хеша.

In [32]:
class MultivalentEncoder(nn.Module):
    def __init__(self, embeddings: nn.Embedding):
        super().__init__()
        self.emb = embeddings

    def forward(self, ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        num_hashes = ids.shape[1]
        ans = []

        for i in range(num_hashes):
            now = ids[:, i]

            off = torch.cumsum(lengths, 0)
            off = torch.cat(
                (
                    torch.tensor([0], device=lengths.device),
                    off
                )
            )
            off = off[:-1]

            res = F.embedding_bag(
                now,
                self.emb.weight,
                offsets=off
            )

            ans.append(res)

        stacked = torch.stack(ans)
        return stacked.permute(1, 0, 2)

## PiecewiseLinearEncoder

Теперь нужно реализовать `PiecewiseLinearEncoder` — слой для кусочно-линейного кодирования вещественных признаков как в статье [On Embeddings for Numerical Features in Tabular Deep Learning](https://arxiv.org/abs/2203.05556).

Идея этого преобразования такая: каждый вещественный признак разбивается на интервалы по квантилям, а затем значение признака представляется набором кусочно-линейных активаций относительно этих интервалов. В результате один вещественный признак превращается в несколько чисел, которые уже удобнее подавать в табличную нейросетевую модель.

В этом задании вам нужно реализовать:
- `compute_bins`;
- `from_dataset`;
- `__init__`;
- свойство `n_bins`;
- `forward`.

---

### Вход и выход слоя

На вход `forward` подаётся тензор

```python
[batch_size, n_features]
```

где:
- batch_size — число объектов в батче;
- n_features — число вещественных признаков.

На выходе должен получаться тензор

```python
[batch_size, encoded_dim]
```

где encoded_dim — суммарная размерность кусочно-линейного представления всех признаков.

---

Что делает `compute_bins`

Метод `compute_bins(X, n_bins)` должен:
1.	принять тензор X формы `[n_objects, n_features]`;
2.	для каждой колонки отдельно посчитать квантили;
3.	вернуть список тензоров с границами бинов, по одному тензору на каждый признак.

Если бы у всех квантилей были разные значения, то для каждого признака получилось бы ровно n_bins + 1 границ. Но на практике это не всегда так: если в колонке много одинаковых значений, некоторые квантили совпадут. Поэтому после вычисления квантилей нужно удалить повторы через `unique()`.

Именно поэтому реальное число бинов у признака может быть:
- равно n_bins;
- меньше n_bins;
- в крайнем случае равно 1.

---

Что возвращает compute_bins

Метод должен вернуть список длины n_features:

```python
[
    tensor([...]),  # границы бинов для 1-го признака
    tensor([...]),  # границы бинов для 2-го признака
    ...
]
```

Для признака с k бинами тензор границ имеет длину k + 1.

Например:
- если после удаления дублей осталось 5 границ, то это означает 4 бина;
- если осталось 2 границы, то это означает 1 бин.

---

Что делает `from_dataset`

Метод `from_dataset(...)` должен построить готовый объект PiecewiseLinearEncoder по обучающим данным.

Для этого нужно:
1.	взять срез обучающего датафрейма dense_train_df (например, 1_000_000 первых строк);
2.	преобразовать его в тензор;
3.	посчитать биновые границы через compute_bins;
4.	по этим границам построить все внутренние параметры энкодера;
5.	вернуть готовый экземпляр класса.

---

Что такое `weight` и `bias`

Для каждого признака и каждого бина мы заранее строим коэффициенты линейной функции. Если границы очередного бина равны

$$
[a, b],
$$

то для него считаются коэффициенты

$$
w = \frac{1}{b-a}, \qquad c = -\frac{a}{b-a}.
$$

Тогда линейная функция на этом бине имеет вид

$$
w x + c = \frac{x-a}{b-a}.
$$

Именно поэтому всё преобразование потом можно считать векторизованно через

$$
bias + weight \odot x.
$$

Здесь:
- weight хранит наклоны линейных функций;
- bias хранит сдвиги.

---

Почему число бинов может отличаться у разных признаков

После удаления одинаковых квантилей у разных колонок может остаться разное число границ, а значит и разное число бинов.

Например:
- у первого признака может остаться 4 бина;
- у второго — только 2;
- у третьего — только 1.

Чтобы хранить всё в одном тензоре, нужно:
1.	взять максимальное число бинов max_n_bins;
2.	завести weight и bias размера `[n_features, max_n_bins]`;
3.	для признаков с меньшим числом бинов часть позиций оставить пустыми;
4.	затем в forward убрать лишние координаты с помощью mask.

---

Что делает `mask`

`mask` нужен для случая, когда у разных признаков разное число бинов.

Если у всех признаков число бинов одинаковое, то mask можно не использовать и оставить None.

Если число бинов различается, то:
- представление приходится паддить до общего max_n_bins;
- после этого mask указывает, какие координаты нужно оставить, а какие удалить.

Итоговая логика такая:
- сначала строится представление размера [batch_size, n_features, max_n_bins];
- потом оно разворачивается;
- потом лишние паддинговые координаты удаляются через mask.

---

Что делает `single_bin_mask`

single_bin_mask нужен для признаков, у которых после удаления одинаковых квантилей остался ровно один бин.

Это специальный крайний случай. Для таких признаков последняя координата в forward обрабатывается не совсем так же, как для остальных. Поэтому нужен отдельный булевый вектор длины n_features, который отмечает признаки с единственным бином.

Если таких признаков нет, single_bin_mask можно оставить None.

---

Какие крайние случаи нужно учитывать

1. У признака меньше бинов, чем n_bins
Это нормальная ситуация. Она возникает, когда часть квантилей совпала из-за повторяющихся значений. Ничего дополнительно делать не нужно: просто работаем с тем числом бинов, которое реально получилось после `unique()`.

2. У признака ровно один бин
Это тоже допустимая ситуация. Она означает, что после удаления повторов у признака осталось ровно две разные границы.

Такой признак не нужно выбрасывать. Его нужно корректно обработать через single_bin_mask.

3. У признака вообще нет интервала
Если после вычисления квантилей и удаления дублей у признака остаётся только одна уникальная граница, то это означает, что колонка по сути константная и для неё невозможно построить даже один бин.

В этом случае нужно выбросить ошибку:

```python
assert n_bin >= 1, "There is a column with only one unique value"
```

Иными словами:
- 1 бин — допустимо;
- 0 бинов — недопустимо.

4. У всех признаков одинаковое число бинов
В этом случае mask не нужен, и его можно хранить как None.

5. У разных признаков разное число бинов
В этом случае mask обязателен, иначе на выходе останутся лишние координаты от паддинга.

---

Что нужно сделать в `__init__`

В конструкторе нужно:
1.	сохранить список n_bins;
2.	сохранить `weight` и `bias` как buffer;
3.	сохранить mask как buffer, если он не None;
4.	сохранить single_bin_mask как buffer, если он не None.

Здесь именно register_buffer, а не nn.Parameter, потому что это не обучаемые параметры, а заранее посчитанные служебные тензоры.

---

Что должно возвращать свойство `n_bins`

Свойство n_bins должно возвращать список числа бинов для каждого вещественного признака.

Например, если у трёх признаков получилось 4, 2 и 1 бин, то:

```python
encoder.n_bins == [4, 2, 1]
```

---

Что нужно сделать в forward

В forward(x) нужно:
1.	взять вход x формы [batch_size, n_features];
2.	применить векторизованное преобразование через bias и weight;
3.	правильно обрезать значения:
	- первая координата должна быть ограничена сверху;
	- внутренние координаты должны быть ограничены с двух сторон;
	- последняя координата должна обрабатываться отдельно;
4.	учесть специальный случай признаков с одним бином через single_bin_mask;
5.	развернуть последние две размерности;
6.	если есть mask, применить его;
7.	вернуть итоговый тензор признаков.

---

Интуиция про обрезание значений

После вычисления линейных функций получается набор чисел, который нужно превратить в корректное кусочно-линейное представление.

Для этого:
- левая часть должна насыщаться сверху;
- внутренние части должны лежать в диапазоне от 0 до 1;
- правая часть должна насыщаться снизу;
- для признаков с одним бином последняя координата обрабатывается отдельно.

Именно поэтому в forward используются `clamp`, `clamp_min` и `clamp_max`.

---

Подсказка:
- Если не получается решить задачу, то можно обратиться к оригинальной реализации от авторов: https://github.com/yandex-research/rtdl-num-embeddings/tree/a8fc25025c83f2321c63ff127a3bcef83bb1bfb5


In [21]:
class PiecewiseLinearEncoder(nn.Module):
    @staticmethod
    def compute_bins(
        X: torch.Tensor,
        n_bins: int,
    ) -> list[torch.Tensor]:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        bin_edges = torch.quantile(
            X, torch.linspace(0, 1, n_bins + 1, device=X.device), dim=0
        )

        bins_per_feature = []
        for feature_idx in range(bin_edges.shape[1]):
            unique_edges = torch.unique(bin_edges[:, feature_idx])
            bins_per_feature.append(unique_edges)

        return bins_per_feature

    @classmethod
    def from_dataset(
        cls: "PiecewiseLinearEncoder",
        dense_train_df: pl.DataFrame,
        n_bins: int = 32,
        train_df_slice: int = 1_000_000,
    ) -> "PiecewiseLinearEncoder":
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        dense_trian_tensor = torch.tensor(
            dense_train_df.head(train_df_slice).to_numpy(), dtype=torch.float32
        )

        bins_per_feature = cls.compute_bins(dense_trian_tensor, n_bins)

        n_features = len(bins_per_feature)
        max_n_bins = max(len(b) - 1 for b in bins_per_feature)

        weight = torch.zeros(n_features, max_n_bins, dtype=torch.float32)
        bias = torch.zeros(n_features, max_n_bins, dtype=torch.float32)

        mask = torch.zeros(n_features, max_n_bins, dtype=torch.bool)
        single_bin_mask = torch.zeros(n_features, dtype=torch.bool)

        n_bins_list = []

        for feat_idx, edges in enumerate(bins_per_feature):
            n_bins = len(edges) - 1
            n_bins_list.append(n_bins)

            assert n_bins >= 1, f"Feature {feat_idx} has only one unique value."

            for bin_idx in range(n_bins):
                a, b = edges[bin_idx], edges[bin_idx + 1]
                width = (b - a).item()
                weight[feat_idx, bin_idx] = 1.0 / max(width, 1e-8)
                bias[feat_idx, bin_idx] = -a / max(width, 1e-8)

            mask[feat_idx, :n_bins] = True
            single_bin_mask[feat_idx] = n_bins == 1

        return cls(weight, bias, mask, n_bins_list, single_bin_mask)

    def __init__(
        self,
        weight: torch.Tensor,
        bias: torch.Tensor,
        mask: torch.Tensor,
        n_bins: int,
        single_bin_mask,
    ):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        super().__init__()

        self.weight = nn.Buffer(weight)
        self.bias = nn.Buffer(bias)
        self.single_bin_mask = nn.Buffer(single_bin_mask)
        self.mask = nn.Buffer(mask)
        self._n_bins = n_bins

    @property
    def n_bins(self):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        return self._n_bins

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        encoded = self.bias + self.weight * x.unsqueeze(-1)

        if encoded.size(-1) >= 1:
            encoded[..., 0] = torch.clamp(encoded[..., 0], max=1.0)

        if encoded.size(-1) >= 2:
            if encoded.size(-1) > 2:
                encoded[..., 1:-1] = torch.clamp(
                    encoded[..., 1:-1], min=0.0, max=1.0
                )
            encoded[..., -1] = torch.clamp(encoded[..., -1], min=0.0)

        encoded[:, self.single_bin_mask, 0] = encoded[
            :, self.single_bin_mask, 0
        ].clamp(0.0, 1.0)

        flat_encoded = encoded.flatten(start_dim=1)

        flat_mask = self.mask.flatten()
        result = flat_encoded[:, flat_mask]

        return result

## DeepNetwork

Теперь реализуем `DeepNetwork` — обычную полносвязную сеть из линейных слоёв и `ReLU`.

В конструктор передаются:
- `input_dim` — размер входа;
- `hidden_units` — список размеров скрытых слоёв.

Для каждого значения из `hidden_units` нужно добавить `Linear`, а затем `ReLU`. В `forward` нужно просто применить получившуюся сеть к входу `x`.

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход должен иметь форму

```python
[batch_size, hk]
```

In [22]:
class DeepNetwork(nn.Module):
    def __init__(self, input_dim, hidden_units):
      super().__init__()
      dims = [input_dim] + hidden_units
      layers = []
      for i in range(1, len(dims)):
        layers.append(nn.Linear(dims[i - 1], dims[i]))
        layers.append(nn.ReLU())
      self.network = nn.Sequential(*layers)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
      return self.network(x)

## ConcatMLP

Теперь нужно реализовать модель `ConcatMLP` — первую полную нейросетевую модель в этом задании.

Идея модели такая:
- sparse-признаки кодируются через `CategoricalEncoder`;
- multivalent-признаки кодируются через `MultivalentEncoder`;
- вещественные признаки кодируются через `PiecewiseLinearEncoder`;
- все полученные представления конкатенируются в один вектор;
- этот вектор подаётся в `DeepNetwork`;
- поверх выхода `DeepNetwork` применяется линейный слой, который выдаёт итоговый score.

### Что передаётся в конструктор

- `embedding_size` — размер эмбеддингов для категориальных и мультивалентных признаков;
- `deep_units` — размеры скрытых слоёв `DeepNetwork`;
- `input_size` — размер входа в `DeepNetwork` после конкатенации всех представлений;
- `dense_train_df` — обучающие вещественные признаки, по которым нужно построить `PiecewiseLinearEncoder`;
- `n_bins` — число бинов для кусочно-линейного кодирования;
- `train_df_slice` — размер среза датафрейма, на котором строится `PiecewiseLinearEncoder`;
- `cardinality` — размер общего embedding-словаря;
- `output_size` — размер выхода модели.

### Что нужно сделать в `__init__`

1. Создать общий слой `nn.Embedding`.
2. Создать:
   - `CategoricalEncoder`,
   - `MultivalentEncoder`,
   - `PiecewiseLinearEncoder`,
   - `DeepNetwork`,
   - финальный `Linear`.
3. Сохранить все эти модули в поля класса.

### Что приходит в `forward`

На вход подаётся словарь `inputs` с той же структурой, что и в `RankerDataset`:

- `inputs["dense_features"]`
- `inputs["sparse_features"]["item_id"]`
- `inputs["sparse_features"]["uid"]`
- `inputs["multivalent_features"]["artist_ids"]["values"]`
- `inputs["multivalent_features"]["artist_ids"]["lengths"]`
- `inputs["multivalent_features"]["album_ids"]["values"]`
- `inputs["multivalent_features"]["album_ids"]["lengths"]`

### Что нужно сделать в `forward`

1. Закодировать `item_id` и `uid` через `CategoricalEncoder`.
2. Закодировать `artist_ids` и `album_ids` через `MultivalentEncoder`.
3. Закодировать dense-признаки через `PiecewiseLinearEncoder`.
4. Развернуть категориальные и мультивалентные представления по последним размерностям.
5. Сконкатенировать все представления по последней оси.
6. Передать результат в `DeepNetwork`.
7. Применить финальный линейный слой.
8. Вернуть итоговый output.

### Ожидаемое поведение

Если на вход подан батч размера `batch_size`, то выход модели должен иметь форму

```python
[batch_size, output_size]

In [23]:
class ConcatMLP(nn.Module):
    def __init__(
        self,
        embedding_size: int,
        deep_units,
        sparse_input_size: int,
        dense_train_df: pl.DataFrame,
        n_bins: int,
        train_df_slice: pl.DataFrame,
        cardinality: int = 65536,
        output_size: int = 1,
    ):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        super().__init__()

        self.embedding = nn.Embedding(cardinality, embedding_size)
        self.categorical_encoder = CategoricalEncoder(self.embedding)
        self.multivalent_encoder = MultivalentEncoder(self.embedding)

        self.dense_encoder = PiecewiseLinearEncoder.from_dataset(
            dense_train_df,
            n_bins=n_bins,
            train_df_slice=train_df_slice,
        )

        self.deep = DeepNetwork(
            sparse_input_size + sum(self.dense_encoder.n_bins), deep_units
        )
        self.output = nn.Linear(deep_units[-1], output_size)

    def forward(self, inputs: dict) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        full_input = []
        full_input.append(self.dense_encoder(inputs["dense_features"]))

        for sparse_feature in ["uid", "item_id"]:
            full_input.append(
                self.categorical_encoder(
                    inputs["sparse_features"][sparse_feature]
                ).flatten(start_dim=1)
            )

        for multivalent_feature in ["artist_ids", "album_ids"]:
            full_input.append(
                self.multivalent_encoder(
                    inputs["multivalent_features"][multivalent_feature][
                        "values"
                    ],
                    inputs["multivalent_features"][multivalent_feature][
                        "lengths"
                    ],
                ).flatten(start_dim=1)
            )

        x = torch.cat(full_input, dim=1)

        x = self.deep(x)
        x = self.output(x)

        return x


## to_device

Теперь нужно реализовать функцию `to_device`, которая рекурсивно переносит все тензоры в произвольной вложенной структуре данных на заданное устройство.

Такая функция понадобится дальше, чтобы удобно переносить целый батч на CPU или GPU, не разбирая вручную каждое поле.

### Что приходит на вход

- `obj` — произвольный объект;
- `device` — устройство, на которое нужно перенести тензоры.

`obj` может быть:
- `torch.Tensor`;
- `dict`;
- `list`;
- `tuple`;
- любым другим объектом.

### Что нужно сделать

- если `obj` — это `torch.Tensor`, нужно вернуть `obj.to(device)`;
- если `obj` — это `dict`, нужно рекурсивно применить `to_device` ко всем значениям;
- если `obj` — это `list` или `tuple`, нужно рекурсивно применить `to_device` ко всем элементам и сохранить исходный тип контейнера;
- для всех остальных объектов нужно просто вернуть их без изменений.

### Зачем это нужно

Батч в этом задании представляет собой вложенный словарь с dense-, sparse-, multivalent-признаками, таргетами и мета-информацией. Чтобы перед обучением перенести такой батч на нужное устройство, удобно иметь одну универсальную функцию, которая умеет делать это рекурсивно.

In [24]:
def to_device(obj, device: torch.device | str):
    if isinstance(obj, torch.Tensor):
      return obj.to(device)
    elif isinstance(obj, dict):
      return {k: to_device(v, device) for k, v in obj.items()}
    elif isinstance(obj, list):
      return [to_device(x, device) for x in obj]
    elif isinstance(obj, tuple):
      return tuple(to_device(x, device) for x in obj)
    else:
      return obj

## Train Loop

Далее вам нужно написать train loop, в котором будет происходить обучение вашей модели. Добавьте в него подсчёт `pairwise accuracy` на тестовом наборе данных после каждой эпохи.

Также предусмотрите поддержку `output_size=1` и `output_size=2`. Это нужно, потому что далее модель будет решать сразу две задачи:
- предсказание лайка;
- предсказание полного прослушивания.

**В случае `output_size=1` обучаемся только на предсказание вероятности полного прослушивания. В случае `output_size=2` на лайк и полное прослушивание.**

Каждая задача должна иметь одинаковый вклад в общую функцию потерь, поэтому никаких дополнительных перевзвешиваний добавлять не нужно. Используйте `BCEWithLogitsLoss`.

В случае `output_size=1` нужно вернуть:

```python
{
    "pair_accuracy_full_play": ...,
    "pair_accuracy_full_play_like_pairs": ....
}
```

при `output_size=2` нужно вернуть

```python
{
    "pair_accuracy_full_play": ...,
    "pair_accuracy_full_play_like_pairs": ...
    "pair_accuracy_like": ...,
    "pair_accuracy_like_full_play_pairs": ...
}
```

In [25]:
from tqdm.auto import tqdm


def evaluate_epoch(
    model: nn.Module,
    test_loader: torch.utils.data.DataLoader,
    device: torch.device,
):
    model.eval()

    collected = defaultdict(list)
    type_to_columns = {
        "meta": ["uid", "timestamp"],
        "labels": ["is_like", "is_full_play"],
    }

    with torch.inference_mode():
        for batch in test_loader:
            for _type, columns in type_to_columns.items():
                for column in columns:
                    collected[column].append(batch[_type][column])

            batch = to_device(batch, device)
            with torch.amp.autocast("cuda", torch.bfloat16):
                predictions = model(batch)
                predictions = predictions.float()

            collected["predictions"].append(predictions.cpu())

    concatenated = {}
    for key, value in collected.items():
        concatenated[key] = torch.cat(value).numpy()

    metrics = {}

    def compute_metric_wrapper(
        main_metric: str, pair_metric: str, predictions: np.ndarray
    ):
        metrics[f"pair_accuracy_{main_metric}"] = compute_pairwise_accuracy(
            concatenated["uid"],
            concatenated["timestamp"],
            concatenated[f"is_{main_metric}"],
            predictions,
        )
        metrics[f"pair_accuracy_{main_metric}_{pair_metric}_pairs"] = (
            compute_pairwise_accuracy(
                concatenated["uid"],
                concatenated["timestamp"],
                concatenated[f"is_{pair_metric}"],
                predictions,
            )
        )

    if predictions.shape[1] == 1:
        probs = concatenated["predictions"][:, 0]
        compute_metric_wrapper("full_play", "like", probs)
    else:
        like_probs = concatenated["predictions"][:, 0]
        full_probs = concatenated["predictions"][:, 1]

        compute_metric_wrapper("full_play", "like", full_probs)
        compute_metric_wrapper("like", "full_play", like_probs)

    return metrics


def train_model(
    model: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    test_loader: torch.utils.data.DataLoader,
    epochs: int,
    lr: float,
    embed_lr: float,
    train_log_every: int,
):
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    device = "cuda"
    model = model.to(device)

    dense_optimizer = torch.optim.Adam(
        [
            param
            for name, param in model.named_parameters()
            if "embedding" not in name
        ],
        lr=lr,
        fused=True,
    )
    embeddings_optimizer = torch.optim.Adam(
        [
            param
            for name, param in model.named_parameters()
            if "embedding" in name
        ],
        lr=embed_lr,
        fused=True,
    )
    loss_fn = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()

        train_pbar = tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            desc=f"Epoch {epoch} [train]",
            leave=False,
        )
        loss_so_far = 0
        for i, batch in train_pbar:
            batch = to_device(batch, device)

            with torch.amp.autocast("cuda", torch.bfloat16):
                predictions = model(batch)

            if predictions.shape[1] == 1:
                target = batch["labels"]["is_full_play"].unsqueeze(1)
                loss = loss_fn(predictions, target)
            else:
                target = torch.stack(
                    [
                        batch["labels"]["is_like"],
                        batch["labels"]["is_full_play"],
                    ],
                    dim=1,
                )
                loss = loss_fn(predictions, target)

            loss_so_far += loss.detach()

            dense_optimizer.zero_grad()
            embeddings_optimizer.zero_grad()
            loss.backward()
            dense_optimizer.step()
            embeddings_optimizer.step()

            if i % train_log_every == 0:
                train_pbar.set_postfix(
                    {"loss": f"{loss_so_far.item() / (i + 1):.4f}"}
                )

        metrics = evaluate_epoch(model, test_loader, device)
        print(f"\nEpoch {epoch} metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.5f}")

    return metrics

## Обучение

Вам нужно подобрать параметры обучение так, чтобы пройти тест.

In [26]:
CARDINALITY = 2**22

ITEM_ID_SEEDS = [42, 43, 44]
UID_SEEDS = [45, 46]
ATRIST_ID_SEEDS = [41, 40]
ALBUM_IDS_SEEDS = [38, 37]
multihash_transform = MultihashTransform(
    sparse_features_config={
        "item_id": ITEM_ID_SEEDS,
        "uid": UID_SEEDS,
    },
    sparse_features_name="sparse_features",
    multivalent_features_config={
        "artist_ids": ATRIST_ID_SEEDS,
        "album_ids": ALBUM_IDS_SEEDS,
    },
    multivalent_features_name="multivalent_features",
    cardinality=CARDINALITY,
)
TOTAL_SPARSE_FEATURES = (
    len(ITEM_ID_SEEDS)
    + len(UID_SEEDS)
    + len(ATRIST_ID_SEEDS)
    + len(ALBUM_IDS_SEEDS)
)

train_dataset = RankerDataset(
    train_listens,
    [multihash_transform],
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size=512,
)

test_dataset = RankerDataset(
    test_listens,
    [multihash_transform],
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size=1024,
)

train_listens_dense_million = train_listens[DENSE_COLUMNS].slice(0, 1_000_000)
# del train_listens
# del test_listens
gc.collect()

100%|██████████| 909/909 [00:22<00:00, 40.13it/s]


0

In [27]:
from torch.utils.data import DataLoader

common_params = dict(
    batch_size=1,
    num_workers=0,
    pin_memory=False,
    collate_fn=lambda batch: batch[0],
)

train_len = len(train_dataset)
train_loader_small = DataLoader(
    torch.utils.data.Subset(
        train_dataset, range(train_len * 3 // 4, train_len)
    ),
    shuffle=True,
    **common_params,
)
train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **common_params,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **common_params,
)

In [28]:
!python -m pip uninstall -y torch torchvision torchaudio
!python -m pip install -U numpy
!python -m pip install \
  torch==2.10.0 \
  --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached torch-2.10.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
Using cached torch-2.10.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl (916.9 MB)


In [29]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.10.0+cu128
12.8
True


In [33]:
embedding_dim = 16

metrics_concat_mlp = defaultdict(float)
n_runs = 3
for i in range(n_runs):
    model_concat_mlp = ConcatMLP(
        embedding_size=embedding_dim,
        deep_units=[256, 128, 64, 32],
        sparse_input_size=embedding_dim * TOTAL_SPARSE_FEATURES,
        dense_train_df=train_listens_dense_million,
        n_bins=32,
        train_df_slice=1_000_000,
        cardinality=CARDINALITY,
        output_size=1,
    )
    metrics = train_model(
        model_concat_mlp,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=1,
        lr=1e-3,
        embed_lr=3e-4,
        train_log_every=50,
    )
    for key, value in metrics.items():
        metrics_concat_mlp[key] += value

for key, value in metrics_concat_mlp.items():
    metrics_concat_mlp[key] /= n_runs
print(f"averaged {metrics_concat_mlp}")


Epoch 0 metrics:
  pair_accuracy_full_play: 0.58581
  pair_accuracy_full_play_like_pairs: 0.48841



Epoch 0 metrics:
  pair_accuracy_full_play: 0.58702
  pair_accuracy_full_play_like_pairs: 0.48752



Epoch 0 metrics:
  pair_accuracy_full_play: 0.58632
  pair_accuracy_full_play_like_pairs: 0.48901
averaged defaultdict(<class 'float'>, {'pair_accuracy_full_play': 0.5863829432023904, 'pair_accuracy_full_play_like_pairs': 0.4883150971522345})


# 4. DCN-v2 (3 балла)

В этом задании вам нужно реализовать cross-слой из статьи [DCN V2: Improved Deep & Cross Network and Practical Lessons for Web-scale Learning to Rank Systems](https://arxiv.org/pdf/2008.13535) в варианте **Mixture of Low-Rank Experts**.

Кроме того, дополнительно к `DeepNetwork` вам нужно реализовать ещё две архитектуры:
- `ResDeepNetwork`;
- `DenseDeepNetwork`.

После этого вам нужно будет провести небольшой анализ и сравнить качество разных подходов.

## Mixture Low Rank Cross Layer

Вам нужно реализовать `MixtureLowRankCrossLayer` и `MixtureLowRankCrossNetwork` — low-rank вариант cross-слоя из DCN V2 со смесью экспертов.

Идея этого блока такая: входной вектор признаков несколько раз пропускается через специальные cross-слои, которые моделируют явные взаимодействия между признаками. В отличие от обычного полносвязного слоя, здесь новое представление строится через произведение исходного входа `x0` и преобразования текущего состояния `xl`.

В этой реализации каждый cross-слой состоит из смеси low-rank экспертов:
- каждый эксперт задаёт своё low-rank преобразование;
- затем gate-сеть вычисляет веса экспертов;
- итоговое обновление получается как взвешенная сумма выходов всех экспертов.

### Что нужно реализовать в `MixtureLowRankCrossLayer`

В конструкторе нужно:
1. сохранить `input_dim`, `num_experts` и `rank`;
2. создать обучаемые параметры:
   - `U` формы `[num_experts, input_dim, rank]`,
   - `V` формы `[num_experts, input_dim, rank]`,
   - `bias` формы `[input_dim]`;
3. создать `gate` — линейный слой, который по `xl` предсказывает веса экспертов;
4. инициализировать параметры.

В `forward(x0, xl)` нужно:
1. применить low-rank преобразование к `xl` через `V`, а затем через `U`;
2. прибавить `bias`;
3. умножить результат поэлементно на `x0`;
4. посчитать веса экспертов через `softmax(gate(xl))`;
5. смешать выходы экспертов с этими весами;
6. прибавить residual `xl`.

Итоговый выход должен иметь ту же форму, что и вход:
```python
[batch_size, input_dim]
```

Что нужно реализовать в `MixtureLowRankCrossNetwork`

Это просто стек из нескольких MixtureLowRankCrossLayer.

В конструкторе нужно:
1. создать num_layers cross-слоёв;
2. сохранить их в nn.ModuleList.

В forward(x) нужно:
1. сохранить исходный вход как x0;
2. завести текущее состояние xl = x;
3. последовательно прогнать xl через все cross-слои;
4. вернуть результат последнего слоя.

Важная деталь

Во всей сети x0 остаётся фиксированным и всегда равен исходному входу, а xl меняется от слоя к слою.

Ожидаемое поведение
- MixtureLowRankCrossLayer принимает x0 и xl формы [batch_size, input_dim] и возвращает тензор той же формы.
- MixtureLowRankCrossNetwork принимает x формы [batch_size, input_dim] и возвращает тензор той же формы.



In [34]:
class MixtureLowRankCrossLayer(nn.Module):
    def __init__(self, input_dim: int, num_experts: int, rank: int):
        super().__init__()
        self.input_dim = input_dim
        self.num_experts = num_experts
        self.rank = rank
        self.U = nn.Parameter(torch.randn(num_experts, input_dim, rank))
        self.V = nn.Parameter(torch.rand(num_experts, input_dim, rank))
        self.bias = nn.Parameter(torch.rand(input_dim))
        self.gate = nn.Linear(input_dim, num_experts)

    def forward(self, x0: torch.Tensor, xl: torch.Tensor) -> torch.Tensor:
        expert_outputs = []
        for i in range(self.num_experts):
            y = xl @ self.V[i]
            z = y @ self.U[i].T
            z = z + self.bias
            z = x0 * z
            expert_outputs.append(z)

        expert_outputs = torch.stack(expert_outputs, dim=1)
        weights = torch.softmax(self.gate(xl), dim=1).unsqueeze(-1)
        mixed = (expert_outputs * weights).sum(dim=1)
        return mixed + xl


class MixtureLowRankCrossNetwork(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_layers: int,
        num_experts: int,
        rank: int,
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            MixtureLowRankCrossLayer(
                input_dim=input_dim,
                num_experts=num_experts,
                rank=rank,
            )
            for _ in range(num_layers)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x0 = x
        xl = x

        for layer in self.layers:
            xl = layer(x0, xl)
        return xl

## ResDeepNetwork

Теперь вам нужно реализовать `ResidualMLPBlock` и `ResDeepNetwork` — residual-вариант глубокой полносвязной сети.

Идея здесь такая: вместо обычной последовательности `Linear -> ReLU -> Linear -> ReLU` каждый блок дополнительно использует skip connection. Это помогает стабилизировать обучение и облегчает прохождение градиентов через глубокую сеть.

### ResidualMLPBlock

`ResidualMLPBlock` — это residual-блок из двух линейных слоёв.

В конструкторе нужно:
- создать первый линейный слой;
- создать второй линейный слой;
- при необходимости добавить проекцию `proj`, если размер входа `in_dim` не совпадает с размером выхода `out_dim`.

В `forward` нужно:
1. пропустить вход через MLP-ветку;
2. прибавить skip connection;
3. если размеры не совпадают, сначала применить `proj` к входу;
4. после сложения применить `ReLU`.

Если `in_dim == out_dim`, то в skip connection можно использовать вход `x` напрямую. Если размеры отличаются, нужно сначала перевести вход в размерность `out_dim` через линейную проекцию.

### ResDeepNetwork

`ResDeepNetwork` — это последовательность residual-блоков.

В конструкторе передаются:
- `input_dim` — размер входа;
- `hidden_units` — список выходных размерностей residual-блоков.

Нужно построить цепочку блоков так, чтобы каждый следующий блок принимал выход предыдущего. Для этого можно завести размеры

```python
[input_dim] + hidden_units
```

и затем создать блоки между соседними размерностями.

В `forward` нужно просто последовательно прогнать вход через все блоки.

Ожидаемое поведение

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход ResDeepNetwork должен иметь форму

```python
[batch_size, hk]
```

   

In [35]:
class ResidualMLPBlock(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()

        self.first_l = nn.Linear(in_dim, out_dim)
        self.second_l = nn.Linear(out_dim, out_dim)

        if in_dim != out_dim:
            self.proj = nn.Linear(in_dim, out_dim)
        else:
            self.proj = None


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.proj is None:
            skip = x
        else:
            skip = self.proj(x)

        out = self.first_l(x)
        relu = nn.ReLU()
        out = relu(out)
        out = self.second_l(out)

        out = out + skip
        return relu(out)

class ResDeepNetwork(nn.Module):

    def __init__(self, input_dim: int, hidden_units: list[int]):
        super().__init__()
        dims = [input_dim] + hidden_units
        blocks = []
        for i in range(len(dims) - 1):
            blocks.append(ResidualMLPBlock(dims[i], dims[i + 1]))

        self.blocks = nn.ModuleList(blocks)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for block in self.blocks:
            x = block(x)

        return x

## DenseDeepNetwork

Теперь вам нужно реализовать `DenseDeepNetwork` — полносвязную сеть с dense connectivity.

Идея этой архитектуры такая: каждый следующий слой получает на вход не только выход предыдущего слоя, но и все предыдущие представления, включая исходный вход. То есть на каждом шаге вход в слой строится как конкатенация

```python
x, h_1, h_2, ..., h_{i-1}
```

Это позволяет каждому слою напрямую использовать более ранние признаки и уже посчитанные представления.

При этом важно: конкатенация используется только для формирования входа в следующий слой. Выход самого очередного слоя — это обычный тензор размера, заданного в hidden_units, а не накопленная конкатенация всех прошлых выходов.

Что нужно сделать

В конструкторе передаются:
- input_dim — размер исходного входа;
- hidden_units — список размерностей скрытых слоёв.

Нужно:
1. создать последовательность линейных слоёв;
2. учесть, что размер входа в каждый следующий слой растёт, потому что к нему конкатенируются все предыдущие выходы.

В forward нужно:
1. завести список features, который сначала содержит только исходный вход x;
2. на каждом шаге сконкатенировать все тензоры из features;
3. применить очередной линейный слой и ReLU;
4. добавить новый выход в features;
5. вернуть выход последнего слоя.

Ожидаемое поведение

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход должен иметь форму

```pythonj
[batch_size, hk]
```

In [36]:
class DenseDeepNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_units: list[int]):
        super().__init__()
        self.layers = nn.ModuleList()

        in_dim = input_dim
        for out_dim in hidden_units:
            self.layers.append(nn.Linear(in_dim, out_dim))
            in_dim += out_dim

        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = [x]

        for layer in self.layers:
            inp = torch.cat(features, dim=1)
            out = self.relu(layer(inp))
            features.append(out)

        return features[-1]

## DCN V2

Далее вам нужно реализовать итогвую `DCN-v2` модель. Все аналогично `ConcatMLP` модели, но добавляется новый cross-слой, а так же возможность выбрать deep часть. Испольуйте `build_deep_network`.

In [37]:
def build_deep_network(
    input_dim: int,
    hidden_units: list[int],
    deep_type: str = "mlp",
) -> nn.Module:
    """Factory for the deep tower: ``mlp``, ``resnet``, or ``densenet``."""
    if deep_type == "mlp":
        return DeepNetwork(input_dim, hidden_units)
    if deep_type == "resnet":
        return ResDeepNetwork(input_dim, hidden_units)
    if deep_type == "densenet":
        return DenseDeepNetwork(input_dim, hidden_units)
    raise ValueError(f"Unknown deep_type={deep_type!r}, expected 'mlp', 'resnet', or 'densenet'")

In [40]:
class DCNV2(
    nn.Module
):  # it would be cool if authors would provide some base class for both dcnv2 and concat mlp
    # P.S. I am to lazy to refactor original class, so... ctr+c + ctrl+v mostly
    def __init__(
        self,
        embedding_size,
        cross_layers,
        deep_units,
        input_size,
        dense_train_df,
        n_bins,
        train_df_slice,
        cardinality=65536,
        num_experts: int = 4,
        low_rank: int = 32,
        deep_network: str = "mlp",
        output_size: int = 2,
        full_input_size_provided: bool = True,
    ):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        super().__init__()

        self.embedding = nn.Embedding(cardinality, embedding_size)
        self.init_embedding_weights(self.embedding, init_range=0.02)
        self.categorical_encoder = CategoricalEncoder(self.embedding)
        self.multivalent_encoder = MultivalentEncoder(self.embedding)

        self.dense_encoder = PiecewiseLinearEncoder.from_dataset(
            dense_train_df,
            n_bins=n_bins,
            train_df_slice=train_df_slice,
        )

        total_input_size = (
            input_size + sum(self.dense_encoder.n_bins)
            if not full_input_size_provided
            else input_size
        )

        self.cross_network = MixtureLowRankCrossNetwork(
            input_dim=total_input_size,
            num_layers=cross_layers,
            num_experts=num_experts,
            rank=low_rank,
        )

        self.deep = build_deep_network(
            total_input_size, deep_units, deep_network
        )

        self.output = nn.Linear(deep_units[-1] + total_input_size, output_size)

    @staticmethod
    def init_embedding_weights(module: nn.Embedding, init_range: float = 0.02):
        nn.init.trunc_normal_(
            module.weight,
            std=init_range,
            a=-2 * init_range,
            b=2 * init_range,
        )

    def forward(self, inputs: dict) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        full_input = []
        full_input.append(self.dense_encoder(inputs["dense_features"]))

        for sparse_feature in ["uid", "item_id"]:
            full_input.append(
                self.categorical_encoder(
                    inputs["sparse_features"][sparse_feature]
                ).flatten(start_dim=1)
            )

        for multivalent_feature in ["artist_ids", "album_ids"]:
            full_input.append(
                self.multivalent_encoder(
                    inputs["multivalent_features"][multivalent_feature][
                        "values"
                    ],
                    inputs["multivalent_features"][multivalent_feature][
                        "lengths"
                    ],
                ).flatten(start_dim=1)
            )

        x = torch.cat(full_input, dim=1)

        cross_out = self.cross_network(x)
        deep_out = self.deep(x)

        combined = torch.cat([cross_out, deep_out], dim=1)
        output = self.output(combined)

        return output

## Обучение mlp, resnet, densenet с cross-слоями

Ваша задача подобрать параметры для обучения `DCNV2` с `mlp` частью так, чтобы был пройден тест. А так же сравнить этот варинат с `densenet` и `resnet`.

In [41]:
embedding_dim = 16

metrics_dcnv2_mlp = defaultdict(float)
n_runs = 3
for i in range(n_runs):
    model_dcnv2_mlp = DCNV2(
    
        embedding_size=embedding_dim,
        cross_layers=3,
        deep_units=[256, 128, 64, 32],
        input_size=embedding_dim * TOTAL_SPARSE_FEATURES,
        dense_train_df=train_listens_dense_million,
        n_bins=32,
        train_df_slice=1_000_000,
        cardinality=CARDINALITY,
        num_experts=4,
        low_rank=32,
        deep_network="mlp",
        output_size=1,
        full_input_size_provided=False,
    )

    metrics = train_model(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        model_dcnv2_mlp,
        train_loader=train_loader,
        # train_loader=train_loader_small,
        test_loader=test_loader,
        epochs=1,
        lr=1e-3,
        embed_lr=3e-4,
        train_log_every=50,
    )
    for key, value in metrics.items():
        metrics_dcnv2_mlp[key] += value

for key, value in metrics_concat_mlp.items():
    metrics_dcnv2_mlp[key] /= n_runs
print(f"averaged {metrics_dcnv2_mlp}")


Epoch 0 metrics:
  pair_accuracy_full_play: 0.51394
  pair_accuracy_full_play_like_pairs: 0.49985



Epoch 0 metrics:
  pair_accuracy_full_play: 0.50950
  pair_accuracy_full_play_like_pairs: 0.49019



Epoch 0 metrics:
  pair_accuracy_full_play: 0.51421
  pair_accuracy_full_play_like_pairs: 0.49937
averaged defaultdict(<class 'float'>, {'pair_accuracy_full_play': 0.5125487337606924, 'pair_accuracy_full_play_like_pairs': 0.4964687536246826})


In [42]:
metrics_dcnv2_resnet = defaultdict(float)
n_runs = 3
for i in range(n_runs):
    model_dcnv2_resnet = DCNV2(
        embedding_size=embedding_dim,
        cross_layers=3,
        deep_units=[256, 128, 64, 32],  # it may be better to have same sizes +
        # deeper architecture but there is no tests... And there is no *norm
        # Plus it is kind of interesting how will work resnet + densenet
        # with ~the same architecture (and with the same compute limit)
        input_size=embedding_dim * TOTAL_SPARSE_FEATURES,
        dense_train_df=train_listens_dense_million,
        n_bins=32,
        train_df_slice=1_000_000,
        cardinality=CARDINALITY,
        num_experts=4,
        low_rank=32,
        deep_network="resnet",
        output_size=1,
        full_input_size_provided=False,
    )

    metrics = train_model(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        model_dcnv2_resnet,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=1,
        lr=1e-3,
        embed_lr=3e-4,
        train_log_every=50,
    )
    for key, value in metrics.items():
        metrics_dcnv2_resnet[key] += value

for key, value in metrics_concat_mlp.items():
    metrics_dcnv2_resnet[key] /= n_runs
print(f"averaged {metrics_dcnv2_resnet}")


Epoch 0 metrics:
  pair_accuracy_full_play: 0.50026
  pair_accuracy_full_play_like_pairs: 0.49704



Epoch 0 metrics:
  pair_accuracy_full_play: 0.49731
  pair_accuracy_full_play_like_pairs: 0.50316



Epoch 0 metrics:
  pair_accuracy_full_play: 0.51985
  pair_accuracy_full_play_like_pairs: 0.50448
averaged defaultdict(<class 'float'>, {'pair_accuracy_full_play': 0.5058092953196958, 'pair_accuracy_full_play_like_pairs': 0.501559419019757})


In [ ]:
metrics_dcnv2_densenet = defaultdict(float)
n_runs = 3
for i in range(n_runs):
    model_dcnv2_densenet = DCNV2(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        embedding_size=embedding_dim,
        cross_layers=3,
        deep_units=[256, 128, 64, 32],
        input_size=embedding_dim * TOTAL_SPARSE_FEATURES,
        dense_train_df=train_listens_dense_million,
        n_bins=32,
        train_df_slice=1_000_000,
        cardinality=CARDINALITY,
        num_experts=4,
        low_rank=32,
        deep_network="densenet",
        output_size=1,
        full_input_size_provided=False,
    )

    metrics = train_model(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        model_dcnv2_densenet,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=1,
        lr=1e-3,
        embed_lr=3e-4,
        train_log_every=50,
    )
    for key, value in metrics.items():
        metrics_dcnv2_densenet[key] += value

for key, value in metrics_concat_mlp.items():
    metrics_dcnv2_densenet[key] /= n_runs
print(f"averaged {metrics_dcnv2_densenet}")


Epoch 0 metrics:
  pair_accuracy_full_play: 0.51188
  pair_accuracy_full_play_like_pairs: 0.49399



Epoch 0 metrics:
  pair_accuracy_full_play: 0.49219
  pair_accuracy_full_play_like_pairs: 0.50514


Epoch 0 [train]:  36%|███▌      | 4201/11631 [02:26<04:31, 27.34it/s, loss=3507.3447]

# 5. Многоголовость (2 балла)

Одно из важных преимуществ нейросетей состоит в том, что их можно обучать сразу на несколько таргетов. Например, модель может одновременно решать задачу регрессии (предсказывать число секунд прослушивания) и несколько задач классификации (предсказывать лайк, полное прослушивание и другие типы пользовательского фидбека). На практике таких таргетов может быть довольно много, вплоть до десятков (https://blog.reachsumit.com/posts/2023/04/the-twitter-ml-algo/).

Преимущество такого подхода в том, что модель сразу учится строить общее представление объекта, полезное для разных продуктовых сигналов. После этого уже на этапе анализа или A/B-эксперимента можно подбирать веса разных голов и собирать итоговый скор так, чтобы он лучше соответствовал целям продукта.

В этом задании вы обучите двухголовую нейросетевую модель, которая будет одновременно предсказывать:
- лайк;
- полное прослушивание.

После этого вам нужно будет построить парето-фронт и подобрать такие веса для агрегированного score, чтобы он хорошо работал сразу в двух задачах:
- ранжирование по лайкам;
- ранжирование по полным прослушиваниям.

Если удаётся подобрать хорошие веса, это означает, что итоговое ранжирование одновременно хорошо оптимизирует оба сигнала: пользователи и чаще ставят лайки, и чаще дослушивают треки до конца. Именно такой компромисс обычно и важен для бизнеса.

В конце соберите все результаты в одну таблицу и напишите вывод.

## Обучение DCNV2 с двумя головами

Вам нужно подобрать параметры обучения так, чтобы пройти тест. В этом задании `output_size=2`, `deep_network='mlp'`.

In [ ]:
embedding_dim = 16

metrics_dcnv2_mlp = defaultdict(float)
n_runs = 3
for i in range(n_runs):
    model_dcnv2_mlp = DCNV2(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        embedding_size=embedding_dim,
        cross_layers=3,
        deep_units=[256, 128, 64, 32],
        input_size=embedding_dim * TOTAL_SPARSE_FEATURES,
        dense_train_df=train_listens_dense_million,
        n_bins=32,
        train_df_slice=1_000_000,
        cardinality=CARDINALITY,
        num_experts=4,
        low_rank=32,
        deep_network="mlp",
        output_size=1,
        full_input_size_provided=False,
    )

    metrics = train_model(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        model_dcnv2_mlp,
        train_loader=train_loader,
        # train_loader=train_loader_small,
        test_loader=test_loader,
        epochs=1,
        lr=1e-3,
        embed_lr=3e-4,
        train_log_every=50,
    )
    for key, value in metrics.items():
        metrics_dcnv2_mlp[key] += value

for key, value in metrics_concat_mlp.items():
    metrics_dcnv2_mlp[key] /= n_runs
print(f"averaged {metrics_dcnv2_mlp}")


tests.test_model_dcnv2_mlp_metrics(metrics_dcnv2_mlp)

In [ ]:
metrics_dcnv2_resnet = defaultdict(float)
n_runs = 3
for i in range(n_runs):
    model_dcnv2_resnet = DCNV2(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        embedding_size=embedding_dim,
        cross_layers=3,
        deep_units=[256, 128, 64, 32],  # it may be better to have same sizes +
        # deeper architecture but there is no tests... And there is no *norm
        # Plus it is kind of interesting how will work resnet + densenet
        # with ~the same architecture (and with the same compute limit)
        input_size=embedding_dim * TOTAL_SPARSE_FEATURES,
        dense_train_df=train_listens_dense_million,
        n_bins=32,
        train_df_slice=1_000_000,
        cardinality=CARDINALITY,
        num_experts=4,
        low_rank=32,
        deep_network="resnet",
        output_size=1,
        full_input_size_provided=False,
    )

    metrics = train_model(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        model_dcnv2_resnet,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=1,
        lr=1e-3,
        embed_lr=3e-4,
        train_log_every=50,
    )
    for key, value in metrics.items():
        metrics_dcnv2_resnet[key] += value

for key, value in metrics_concat_mlp.items():
    metrics_dcnv2_resnet[key] /= n_runs
print(f"averaged {metrics_dcnv2_resnet}")


In [ ]:
metrics_dcnv2_densenet = defaultdict(float)
n_runs = 3
for i in range(n_runs):
    model_dcnv2_densenet = DCNV2(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        embedding_size=embedding_dim,
        cross_layers=3,
        deep_units=[256, 128, 64, 32],
        input_size=embedding_dim * TOTAL_SPARSE_FEATURES,
        dense_train_df=train_listens_dense_million,
        n_bins=32,
        train_df_slice=1_000_000,
        cardinality=CARDINALITY,
        num_experts=4,
        low_rank=32,
        deep_network="densenet",
        output_size=1,
        full_input_size_provided=False,
    )

    metrics = train_model(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        model_dcnv2_densenet,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=1,
        lr=1e-3,
        embed_lr=3e-4,
        train_log_every=50,
    )
    for key, value in metrics.items():
        metrics_dcnv2_densenet[key] += value

for key, value in metrics_concat_mlp.items():
    metrics_dcnv2_densenet[key] /= n_runs
print(f"averaged {metrics_dcnv2_densenet}")

## Результаты

| метод | pair_accuracy_like | pair_accuracy_full_play |
|-------|-------------------:|------------------------:|
| random |  |  |
| popular |  |  |
| catboost |  |  |
| concatmlp | |  |
| dcnv2_mlp |  |  |
| dcnv2_resnet |  |  |
| dcnv2_densenet |  |  |
| model_multitask_dcnv2_mlp |  |  |

Вывод:

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################



## Строим парето-фронт

В этом задании вам нужно построить парето-фронт для двухголовой модели.

В нашем случае парето-фронт — это множество компромиссов между двумя целями:
- качеством ранжирования лайков;
- качеством ранжирования полных прослушиваний.

Идея такая: у модели есть две головы, одна предсказывает вероятность лайка, а другая — вероятность полного прослушивания. Из этих двух предсказаний можно собрать итоговый score как взвешенную сумму

$$
s = \alpha \cdot P(\text{like}) + (1 - \alpha) \cdot P(\text{full\_play}),
$$

где $\alpha \in [0, 1]$.

Важно, что в реальной системе ранжирование всё равно обычно происходит **по одному итоговому score**. Даже если модель предсказывает сразу несколько сигналов, на этапе выдачи объектов нужно отсортировать их по одному числу. Поэтому после обучения многоголовой модели нужно понять, как именно агрегировать выходы разных голов в единый score. Как раз для этого и строится парето-фронт.

При разных значениях $\alpha$ мы получаем разные итоговые ранжирования, а значит и разные значения метрик:
- `pair_accuracy_like`;
- `pair_accuracy_full_play`.

От вас требуется:
1. перебрать разные значения $\alpha$;
2. для каждого значения посчитать итоговый score;
3. вычислить `pair_accuracy_like` и `pair_accuracy_full_play`;
4. построить scatter plot:
   - по оси OX — `pair_accuracy_like`;
   - по оси OY — `pair_accuracy_full_play`.

После этого посмотрите на получившийся график и напишите, какие выводы можно сделать. Используйте хотя бы 100 точек. Возьмите модель из предыдущего пункта.

In [ ]:
import matplotlib.pyplot as plt

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

Выводы по графику:

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

## Подберите итоговый score

Теперь вам нужно подобрать такое значение $\alpha$, при котором итоговый score удовлетворяет ограничениям в тесте.

Напомним, что итоговый score строится как

$$
s = \alpha \cdot P(\text{like}) + (1 - \alpha) \cdot P(\text{full\_play}).
$$

На это можно смотреть как на выбор итогового продуктового компромисса: насколько система готова разменивать качество по лайкам на качество по полным прослушиваниям. В реальной задаче именно такой выбор и приходится делать при построении финального ранжирования.

В этом задании вам нужно:
1. подобрать значение $\alpha$;
2. посчитать итоговый score с этим весом;
3. убедиться, что он проходит ограничения из теста.

In [ ]:
alpha =

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
}

tests.test_model_multitask_dcnv2_mlp_combined_score_metrics(metrics)